In [2]:
import numpy as np
import skimage.io
import skimage.color
import skimage.filters
import skimage.transform
from skimage import img_as_ubyte
import matplotlib.pyplot as plt
import constants
import IPython.display as d
import json
import os
import time
import torch
import torchvision
from torchvision import transforms, utils, datasets
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Variable
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import shutil

## PREPROCESSING: LABEL REFORMATTING

Converteer de JSON-labels naar tekstbestanden

In [5]:
import json

path = "./data/labels_full_json/"
new_path = "./data/labels_full/"

aantal = len(os.listdir(path))
for filename in os.listdir(path):
    filename = filename.split(".")[0]
    labels = open(f"{path}/{filename}.json")
    data = json.load(labels)
    labels_converted = open(f"{new_path}/{filename}.txt", "a")
    for i in data['shapes']:
                    width = i['points'][1][0] - i['points'][0][0]
                    height = i['points'][1][1] - i['points'][0][1]
                    x_center = i['points'][0][0] + width/2
                    y_center = i['points'][0][1] + height/2
                    labels_converted.write(f"0\t{x_center}\t\
                        {y_center}\t\
                        {width}\t\
                        {height}\n")
    labels_converted.close()
    labels.close()

Normaliseer de coördinaten en groottes van de labels tussen de 0 en 1 (standaardformaat van YOLO)

In [2]:
dir_labels = "./data/data_512/labels_512"
dir_labels_normalized = "./data/data_512/labels_512_normalized"

for filename in os.listdir(dir_labels):
    filename = filename.split(".")[0]
    labels = open(f"{dir_labels}/{filename}.txt")
    labels_normalized = open(f"{dir_labels_normalized}/{filename}.txt", "a")
    lines = [line.rstrip() for line in labels.readlines()]
    for line in lines:
                name, x_center, y_center, width, height = [abs(round(float(s))/constants.WIDTH_PIECE) for s in line.split("\t")]
                labels_normalized.write(f"{name}\t\
                    {x_center}\t\
                    {y_center}\t\
                    {width}\t\
                    {height}\n")
    labels.close()
    labels_normalized.close()

## PREPROCESSING: CROPPING

Manuele horizontale cropping van de trainingdata (1ste benadering is omslachtig, zie automated cropping). Per foto wordt een invoercoördinaat gevraagd aan de gebruiker. Een vaste breedte van 3000 pixels horizontaal wordt gehanteerd. Het resultaat is een omzetting van 7952x5304 naar 3000x5304, met idealiter zoveel mogelijk planten op de bijgesneden afbeelding. Deze stap wordt enkel gedaan om het trainen van YOLO sneller te laten gaan.

In [ ]:
plt.ion()
teller = 1
aantal = len(os.listdir(os.path.join(os.getcwd(), "data\\images_full\\")))
for filename in os.listdir(os.path.join(os.getcwd(), "data\\images_full\\")):
    print(f"{teller}/{aantal}")
    d.clear_output(wait  = True)
    filename = filename.split(".")[0]
    img = skimage.io.imread(f"./data/images_full/{filename}.JPG")
    plt.imshow(img)
    plt.show()
    next = 'n'
    while next == 'n':
        x_begin = int(input("Enter x_begin:"))
        if x_begin < constants.X_END - constants.X_WIDTH_CROPPED:
            d.clear_output(wait  = True)
            plt.close()
            plt.imshow(img)
            plt.axvline(x=x_begin)
            plt.axvline(x=x_begin+constants.X_WIDTH_CROPPED)
            plt.show()
            next = input("next?")
            if next != 'n':
                img_cropped = img[0:constants.Y_END,x_begin:x_begin+constants.X_WIDTH_CROPPED]
                skimage.io.imsave(f"./data/images_vert_crop/{filename}.JPG", img_cropped)
                labels = open(f"./data/labels_full_json/{filename}.json")
                data = json.load(labels)
                labels_cropped = open(f"./data/labels_vert_crop/{filename}.txt", "a")
                for i in data['shapes']:
                    width = i['points'][1][0] - i['points'][0][0]
                    height = i['points'][1][1] - i['points'][0][1]
                    x_center = i['points'][0][0] + width/2 - x_begin
                    y_center = i['points'][0][1] + height/2
                    labels_cropped.write(f"0\t{x_center}\t\
                        {y_center}\t\
                        {width}\t\
                        {height}\n")
                labels_cropped.close()
                labels.close()
                teller += 1
        

Knip hoge resolutie afbeeldingen in een voorgedefinieerde map in stukken van een gewenste grootte. De daarbijhorende labels worden mee aangepast. Dit wordt gebruikt om de modellen te trainen.

In [ ]:
dir_imgs = "./data/images_vert_crop"
dir_labels = "./data/labels_vert_crop"
x_length = constants.X_WIDTH_CROPPED #512 or 600


aantal = len(os.listdir(os.path.join(os.getcwd(), dir_imgs)))
teller_files = 1
for filename in os.listdir(os.path.join(os.getcwd(), dir_imgs)):
    print(f"{teller_files}/{aantal}")
    filename = filename.split(".")[0]
    #Per image
    img = skimage.io.imread(f"{dir_imgs}/{filename}.JPG")
    labels = open(f"{dir_labels}/{filename}.txt")
    lines = labels.readlines()
    lines = [line.rstrip() for line in lines]
    teller = 1
    y = 0
    while(y < constants.Y_END - constants.WIDTH_PIECE): #zijkanten worden weggegooid indien te klein
        x = 0
        while(x < x_length - constants.WIDTH_PIECE): #zijkanten worden weggegooid indien te klein
            x_end = x+constants.WIDTH_PIECE if x+constants.WIDTH_PIECE < x_length else x_length
            y_end = y+constants.WIDTH_PIECE if y+constants.WIDTH_PIECE < constants.Y_END else constants.Y_END
            print(f"x: {x}\ty: {y}\tx_end: {x_end}\ty_end: {y_end}")
            img_piece = img[y:y_end,x:x_end]
            labels_piece = open(f"./data/data_512/labels_512_yolo/{filename}_{teller}.txt", "a")
            for line in lines:
                name, x_center, y_center, width, height = [round(float(s)) for s in line.split("\t")]
                if x_center >= x and x_center < x_end and y_center >= y and y_center < y_end:
                    labels_piece.write("0\t"+str(x_center-x)+"\t"+str(y_center-y)+"\t"+str(width)+"\t"+str(height)+"\n")
            labels_piece.close()
            if os.stat(labels_piece.name).st_size == 0: os.remove(labels_piece.name)
            print(img_piece)
            skimage.io.imsave(f"./data/data_512/images_512_yolo/{filename}_{teller}.JPG", img_piece)
            x += constants.WIDTH_PIECE - constants.OVERLAP #+550
            teller += 1
        y += constants.WIDTH_PIECE - constants.OVERLAP
    labels.close()
    teller_files+= 1
    


Variant van het vorig script om GT voor baseline training te maken. De labels worden ineens omgezet naar positief of negatief

In [ ]:
dir_imgs = "./data/images_test"
dir_labels = "./data/labels_full"
x_length = 7952 #or constants.X_WIDTH_CROPPED

aantal = len(os.listdir(os.path.join(os.getcwd(), dir_imgs)))
teller_files = 1
for filename in os.listdir(os.path.join(os.getcwd(), dir_imgs)):
    print(f"{teller_files}/{aantal}")
    filename = filename.split(".")[0]
    #Per image
    img = skimage.io.imread(f"{dir_imgs}/{filename}.JPG")
    gt = np.zeros(10168)
    labels = open(f"{dir_labels}/{filename}.txt")
    for line in labels:
        arr = line.split("\t")
        x_center = float(arr[1])
        y_center = float(arr[2])
        index = int(x_center//64 + (x_length//64) * (y_center//64))
        try:
            gt[index] = 1
        except:
            print(filename)
            print(line)
            continue
    teller = 0
    y = 0
    while(y < constants.Y_END - 64): #zijkanten worden weggegooid indien te klein
        x = 0
        while(x < x_length - 64): #zijkanten worden weggegooid indien te klein
            x_end = x+64 if x+64 < x_length else x_length
            y_end = y+64 if y+64 < constants.Y_END else constants.Y_END
            img_piece = img[y:y_end,x:x_end]
            if gt[teller] == 1:
                if teller < 0:
                    skimage.io.imsave(f"./data/data_512/images_64_base/test/positive/{filename}_{teller}.JPG", img_piece)
                else:
                    skimage.io.imsave(f"./data/data_512/images_64_base/train/positive/{filename}_{teller}.JPG", img_piece)
            else:
                if teller < 0:
                    skimage.io.imsave(f"./data/data_512/images_64_base/test/negative/{filename}_{teller}.JPG", img_piece)
                else:
                    skimage.io.imsave(f"./data/data_512/images_64_base/train/negative/{filename}_{teller}.JPG", img_piece)
            x += 64 - constants.OVERLAP #+550
            teller += 1
        y += 64 - constants.OVERLAP
    labels.close()
    teller_files+= 1

(TEST) Test if labels are correctly transformed (absolute coordinates!!)

In [ ]:
img = skimage.io.imread(f"./data/data_512/images_512_yolo/DSC00174_37.JPG")
labels = open(f"./data/data_512/labels_512_yolo/DSC00174_37.txt")
lines = labels.readlines()
lines = [line.rstrip() for line in lines]
line = lines[2]
name, x_center, y_center, width, height = [round(float(s)) for s in line.split("\t")]
img_box = img[round(y_center-height/2):round(y_center+height/2),round(x_center-width/2):round(x_center+width/2)]
print(img_box)
skimage.io.imsave(f"./data//{x_center}.JPG", img_box)

(TEST) Test if labels are correctly transformed (relative coordinates!!)

In [9]:
img = skimage.io.imread(f"./data/data_512/images_512_GT/train/positive/DSC00174_8.JPG")
labels = open(f"./data/data_512/images_512_GT/GT/DSC00174_8.txt")
width_piece = 512
lines = labels.readlines()
lines = [line.rstrip() for line in lines]
line = lines[0]
name, x_center, y_center, width, height = [float(s) for s in line.split("\t")]
img_box = img[round(width_piece*(y_center-height/2)):round(width_piece*(y_center+height/2)),round(width_piece*(x_center-width/2)):round(width_piece*(x_center+width/2))]
skimage.io.imsave(f"./data//{x_center}.JPG", img_box)
labels.close()

(TEST) Test labels for folder

In [ ]:
dir = "./data/data_512/images_1-4_256"
dir_labels = "./data/data_512/labels_1-4"
width_piece = 256

aantal = len(os.listdir(dir))
teller_files = 0

for filename in os.listdir(dir):
    name_short = filename.split(".")[0]
    img = skimage.io.imread(f"{dir}/{name_short}.JPG")
    try:
        labels = open(f"{dir_labels}/{name_short}.txt")
        lines = labels.readlines()
        lines = [line.rstrip() for line in lines]
        line = lines[0]
        name, x_center, y_center, width, height = [float(s) for s in line.split("\t")]
        img_box = img[round(width_piece*(y_center-height/2)):round(width_piece*(y_center+height/2)),round(width_piece*(x_center-width/2)):round(width_piece*(x_center+width/2))]
        skimage.io.imsave(f"./data/test_labels/{x_center}.JPG", img_box)
        teller_files += 1
        if teller_files % 1000 == 0:
            print(f"{teller_files}/{aantal}")
        labels.close()
    except:
        continue

Ligging kevers in afbeelding

In [9]:
import csv
dir = "./data/labels_full"
with open('detectie_cluster_kevers.csv', mode='w', newline='') as file:
    writer = csv.writer(file, delimiter=',', quotechar='"', quoting=csv.QUOTE_MINIMAL)
    writer.writerow(['x', 'y'])
    for filename in os.listdir(dir):
        labels = open(f"{dir}/{filename}")
        lines = [line.rstrip() for line in labels.readlines()]
        writer.writerow([filename])
        for l in lines:
            name, x_center, y_center, width, height = [float(s) for s in l.split("\t")]
            writer.writerow([x_center, y_center])


Snelheid imagesplit

In [13]:
def split_image(img, size):
    data = []
    y = 0
    while(y < constants.Y_END - size):
        x = 0
        while(x < constants.X_END - size):
            x_end = x+size if x+size < constants.X_END else constants.X_END
            y_end = y+size if y+size < constants.Y_END else constants.Y_END
            data.append(img[y:y_end,x:x_end])
            x += size - constants.OVERLAP
        y += size - constants.OVERLAP
    return data

img = skimage.io.imread(f"./data/images_full/DSC00174.JPG")
start = time.time()
_ = split_image(img, 64)
tijd = time.time() - start
print(tijd)

0.0189971923828125
